<h1 style="text-align: center; font-family: 'menlo'; color: #ADD8E6; font-size:50px;">
  <span style="background-color: #191970; padding: 5px 10px; border-radius: 5px; display: inline-block;">
         Reading images
  </span>
</h1>

Astropy provides a few ways to read in FITS images, some in the core package and others in affiliated packages.

Before exploring those, we’ll create a set of (fake) images to work with.

In [2]:
from pathlib import Path #manipular caminhos de pastas/arquivos
from astropy.nddata import CCDData #ler imagens FITS como objetos de CCD 
from astropy.io import fits #módulo básico do astropy pra abrir arquivos FITS

# Working with directories

The cell below contains the path to the images. In this notebook we’ll use it both to store the fake images we generate and to read images. In normal use, you wouldn’t start by writing images there, however.

If the images are in the same directory as the notebook, you can omit this or set it to an empty string ''. Having images in the same directory as the notebook is less complicated, but it’s not at all uncommon to need to work with images in a different directory.

Later, we’ll look at how to generate the full path to an image (directory plus file name) in a way that will work on any platform. One of the approaches to loading images (using <span style="color: #d62495">ccdproc.ImageFileCollection</span>) lets you mostly forget about this.

In [3]:
data_directory = 'path/to/my/images'

# Generate some fake images

The cells below generate some fake images to use later in the notebook.

In [19]:
from pathlib import Path
from itertools import cycle #importa a função cycle, que cria um iterador que repete infinitamente uma sequência
import numpy as np


data_directory = 'fake_images' #pasta dentro do diretório atual do notebook, p funcionar a célula da frente
image_path = Path(data_directory) #converte data_directory em um objeto Path, permitindo manipular melhor o caminho
image_path.mkdir(parents=True, exist_ok=True) #cria o diretório onde as imagens serão salvas,, parents = cria os diretórios pais se não existirem,, ok = não dá erro se o diretório já existir

images_to_generate = {'BIAS': 5,
                     'DARK': 10,
                     'FLAT':3,
                     'LIGHT': 10}

exposure_times = {'BIAS': [0.0],
                 'DARK': [5.0, 30.0],
                 'FLAT': [5.0, 6.1, 7.3],
                 'LIGHT': [30.0]}
#tempo de exposição em segundos

filters = {'FLAT': 'V', 'LIGHT': 'V'}
#filtro V da banda visial, só esses dois tem

objects = {'LIGHT': ['m82', 'xx cyg']}
#só imagens light tem objeto associado, m82 é uma galáxia e xxcyg é uma estrela

image_size = [300, 200]
image_number = 0

for image_type, num in images_to_generate.items():
    exposure = cycle(exposure_times[image_type]) #lista de tempo de exposição pra iterar infinitamente, DARK: 5.0, 30.0, 5.0, 30.0, 5.0... 
    try:
        filts = cycle(filters[image_type]) #ciclo de filtros para esse tipo de imagem
    except KeyError:
        filts = [] #se o tipo nao tiver no dicionario filters (bias e dark), captura o keyerror e define filts como lista vazia

    try: 
        objs = cycle(objects[image_type])
    except KeyError:
        objs = [] #mesma coisa que acima mas com objetos
    for _ in range(num): #loop interno, repete num vezes (quantidade definida no images_to_generate)
        img = CCDData(data=np.random.randn(*image_size), unit='adu') #criação da imagem * desempacota [300, 200] para np.random.randn
        img.meta['IMAGETYP'] = image_type #adicionando os metadados, aqui é header FITS a palavra-chave IMAGETYP com o tipo da imagem (BIAS, DARK, FLAT ou LIGHT)
        img.meta['EXPOSURE'] = next(exposure) #Pega o próximo valor do ciclo de tempos de exposição e coloca no header como EXPOSURE.
        if filts:
            img.meta['FILTER'] = next(filts) #SE houver filtros para esse tipo, pega o próximo filtro do ciclo e coloca no header como FILTER
        if objs:
            img.meta['OBJECT'] = next(objs) #mesma coisa que acima mas SE houver objetos definidos
        image_name = str(image_path / f'img-{image_number:04d}.fits') #montando o nome do arquivo para salvar
        #image_path junta o diretório com o nome do arquivo (qualquer sistema operacional)
        #f'img formata o número com 4 digitos (0000, 0001, 0002...)
        #.fits pra ficar em formato FITS
        img.write(image_name, overwrite=True) #salva imagem no disco no formato FITS
        print(image_name)
        image_number += 1 #imprime o nome do arquivo gerado e imcrementa o contado para o próximo arquivo

fake_images\img-0000.fits
fake_images\img-0001.fits
fake_images\img-0002.fits
fake_images\img-0003.fits
fake_images\img-0004.fits
fake_images\img-0005.fits
fake_images\img-0006.fits
fake_images\img-0007.fits
fake_images\img-0008.fits
fake_images\img-0009.fits
fake_images\img-0010.fits
fake_images\img-0011.fits
fake_images\img-0012.fits
fake_images\img-0013.fits
fake_images\img-0014.fits
fake_images\img-0015.fits
fake_images\img-0016.fits
fake_images\img-0017.fits
fake_images\img-0018.fits
fake_images\img-0019.fits
fake_images\img-0020.fits
fake_images\img-0021.fits
fake_images\img-0022.fits
fake_images\img-0023.fits
fake_images\img-0024.fits
fake_images\img-0025.fits
fake_images\img-0026.fits
fake_images\img-0027.fits


# Option 1: Reading a single image with <span style="color: #d62495">astropy.io.fits</span>

This option gives you the most flexibility but is the <U>least adapted to CCD images</U> specifically. What you read in is a list of FITS extensions; you must first select the one you want then access the data or header as desired.

We’ll open up the first of the fake images, <span style="color: #d62495">img-0001.fits</span>. To combine that with the directory name we’ll use Python 3’s <span style="color: #d62495">pathlib</span>, which ensures that the path combination will work on Windows too.

In [21]:
image_name = 'img-0001.fits' #nome do arquivo antes de abrir
image_path_read = Path(data_directory) / image_name #cria um objeto Path com o caminho da pasta
                                                    # o operador / do pathlib junta o caminho da pasta com o nome do arquivo.

hdu_list = fits.open(image_path_read) #fits vem do astropy.io import fits,,, fits.open() abre o arquivo FITS e retorna um objeto do tipo HDUList.
hdu_list.info() #imprime um resumo de todos os HDUs dentro do arquivo. O output é tipo uma tabelinha.

Filename: fake_images\img-0001.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       8   (200, 300)   float64   


The <span style="color: #d62495">hdu_list</span> is a list of FITS Header-Data Units. In this case there is just one, containing both the image header and data, which can be accessed as shown below.

In [26]:
hdu = hdu_list[0]
#arquivo .fits é como uma caixa que pode conter várias "folhas" (HDUs). Cada folha tem um header (metadados) e um data (array de pixels).
hdu.header

SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -64 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                  200                                                  
NAXIS2  =                  300                                                  
IMAGETYP= 'BIAS    '                                                            
EXPOSURE=                  0.0                                                  
BUNIT   = 'adu     '                                                            

In [49]:
import pandas as pd

hduList = {
    'Name': ['SIMPLE', 'BITPIX', 'NAXIS', 'NAXIS1', 'NAXIS2', 'IMAGETYP', 'EXPOSURE', 'BUNIT'],
    'Output': ['T', -64, 2, 200, 300, 'BIAS', 0.0, 'adu' ],
    'Meaning': ['True, só diz que o arquivo segue o padrão FITS. Sempre é T.',  
                'Float64 (números decimais de 64 bits). Positivo = inteiro.', 
                'A imagem tem 2 dimensões (2D)',
                'Largura da imagem em pixels (eixo X, colunas)', 
                'Altura da imagem em pixels (eixo Y, linhas)',
                'Tipo da imagem. Pode ser BIAS, DARK, FLAT ou LIGHT (science).',
                'Tempo de exposição em segundos. BIAS = 0s (obturador fechado, só lê o detector).',
                'Unidade dos dados: ADU (contagens do detector).']
}

tabela = pd.DataFrame(hduList)

tabela.style.set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).hide(axis='index')

Name,Output,Meaning
SIMPLE,T,"True, só diz que o arquivo segue o padrão FITS. Sempre é T."
BITPIX,-64,Float64 (números decimais de 64 bits). Positivo = inteiro.
NAXIS,2,A imagem tem 2 dimensões (2D)
NAXIS1,200,"Largura da imagem em pixels (eixo X, colunas)"
NAXIS2,300,"Altura da imagem em pixels (eixo Y, linhas)"
IMAGETYP,BIAS,"Tipo da imagem. Pode ser BIAS, DARK, FLAT ou LIGHT (science)."
EXPOSURE,0.000000,"Tempo de exposição em segundos. BIAS = 0s (obturador fechado, só lê o detector)."
BUNIT,adu,Unidade dos dados: ADU (contagens do detector).


In [40]:
hdu.data
#array numpy = shape=(300, 200), dtype='float64'
#cada número é o valor de um pixel. No nosso caso são aleatórios (ruído), mas numa imagem real seriam as contagens de luz que o CCD capturou.

array([[-0.07078007,  0.35714517,  1.17877865, ..., -0.19829086,
        -0.30860744,  0.89953302],
       [ 1.7807912 ,  0.05904263,  0.68323853, ..., -1.44767085,
         0.61681978, -0.39705548],
       [ 0.76616443, -0.61513991, -0.70458446, ..., -0.51094628,
        -1.64175841,  0.13423075],
       ...,
       [-0.55117376, -0.28166224,  1.66047341, ..., -2.23646649,
         0.24036067, -1.16838826],
       [ 1.04648533,  1.12570581, -0.71896989, ...,  0.49011285,
         1.13148937,  2.41325943],
       [-0.52039572,  1.15570148, -0.03323741, ...,  1.59078603,
         0.66050785, -0.84527534]], shape=(300, 200), dtype='>f8')

The [documentation for io.fits](https://docs.astropy.org/en/stable/io/fits/index.html) describes more of its capabilities.

# Option 2: Use <span style="color: #d62495">CCDData</span> to read in a single image

Astropy contains a <span style="color: #d62495">CCDData</span> object for representing a single image. It’s not as flexible as using <span style="color: #d62495">astrop.io.fits</span> directly (for example, it assumes there is only one FITS extension and that it contains image data) but it sets up several properties that make the data easier to work with.

We’ll read in the same single image we did in the example above, <span style="color: #d62495">img-0001.fits.</span>.

<h1 style="text-align: left; font-family: 'menlo'; color: #191970; font-size:40px;">
  <span style="background-color: #ADD8E6; padding: 5px 10px; border-radius: 5px; display: inline-block;">
     Notes
  </span>
</h1>

- Matplotlib [axvline](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.axvline.html) & [pyplot](https://matplotlib.org/stable/tutorials/pyplot.html).

- Usar Path em vez de juntar string: no Windows o separador é \ e no Linux/Mac é /. O pathlib cuida disso sozinho — seu código funciona em qualquer sistema operacional
    - Exemplo: se data_directory = 'fake_images', o resultado é fake_images\img-0001.fits (no Windows) ou fake_images/img-0001.fits (no Linux)

- HDU = Header Data Units, é um arquivo FITS pode conter várias "extensões". Cada extensão tem:
    - Um header (metadados — palavras-chave como EXPOSURE, IMAGETYP, etc.)
    - Um data array (os pixels da imagem em si) 
<br>

- Lembrar que:
    - dsfsdn

---

<h1 style="text-align: left; font-family: 'menlo'; color: #191970; font-size:40px;">
  <span style="background-color: #ADD8E6; padding: 5px 10px; border-radius: 5px; display: inline-block;">
     Links
  </span>
</h1>

https://github.com/astropy/ccd-reduction-and-photometry-guide<br>https://github.com/nyny2903/astropy-coisas/tree/main<br>https://www.astropy.org/ccd-reduction-and-photometry-guide/v/dev/notebooks/01-11-reading-images.html

https://matplotlib.org/stable/users/index.html<br>https://pandas.pydata.org/docs/user_guide/style.html<br>https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.style.html<br>https://pandas.pydata.org/docs/reference/style.html